In [23]:
#Se instala la herramienta RDKit
!pip install rdkit

In [24]:
#Se instala el paquete duckdb que permite manipular la BD que inicialmente es muy grande
!pip install duckdb

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
#Se hace instalación de paquete de kaggle
!pip install -q kaggle

In [35]:
# Se usa la extensión de google colab para subir el token descargado de Kaggle el cual tiene el nombre "kaggle.json"
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"lhcastrop","key":"56e17656944c982f8a2a8576a52759d1"}'}

In [ ]:
#Se crea directorio
!mkdir ~/.kaggle

In [ ]:
#Se copia archivo en carpeta de colab
!cp kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#Se carga el dataset desde kaggle en formato comprimido
! kaggle competitions download -c 'leash-BELKA'

100% 4.16G/4.16G [00:59<00:00, 127MB/s]
100% 4.16G/4.16G [00:59<00:00, 74.9MB/s]


In [ ]:
#Se descomprimen los archivos en el directorio creado
! unzip leash-BELKA.zip -d '/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/Datos_BELKA'

Archive:  leash-BELKA.zip
checkdir:  cannot create extraction directory: /content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/Datos_BELKA
           No such file or directory


**Preparación de Datos**

Los paquetes de datos de entrenamiento y prueba son definidos en los archivos .parquet. Se usa la herramienta duckdb para escanear la busqueda a traves de conjuntos de entramiento grandes. Para comenzar solo se toman un numero igual de muestras positivas y negativas.

Esta busqueda selecciona un numero igual de muestras donde los enlaces son 0 (no-enlazado) y 1 (enlazado), limitado para 30000 cada uno para evitar el sesgo del modelo hacia una clase particular.

In [27]:
from google.colab import drive
drive.mount('/content/drive')
import duckdb
import pandas as pd

train_path = '/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/Datos_BELKA/train.parquet'
test_path = '/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/Datos_BELKA/test.parquet'


con = duckdb.connect()

df = con.query(f"""(SELECT *
                        FROM parquet_scan('{train_path}')
                        WHERE binds = 0
                        ORDER BY random()
                        LIMIT 30000)
                        UNION ALL
                        (SELECT *
                        FROM parquet_scan('{train_path}')
                        WHERE binds = 1
                        ORDER BY random()
                        LIMIT 30000)""").df()

con.close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [28]:
df.head()

,id,buildingblock1_smiles,buildingblock2_smiles,buildingblock3_smiles,molecule_smiles,protein_name,binds
0,152885151,O=C(Nc1c(I)c(C(=O)O)c(I)c(C(=O)O)c1I)OCC1c2ccc...,Cc1cc(N)ccc1Cl,NC[C@]1(CO)COC[C@H]2CCCN21,Cc1cc(Nc2nc(NC[C@]3(CO)COC[C@H]4CCCN43)nc(Nc3c...,BRD4,0
1,292490312,O=C(O)[C@H]1C[C@H](O)CN1C(=O)OCC1c2ccccc2-c2cc...,Cl.Cl.NCc1cnsc1,COc1cnc(N)cn1,COc1cnc(Nc2nc(NCc3cnsc3)nc(N3C[C@@H](O)C[C@@H]...,sEH,0
2,181152225,O=C(Nc1ccc(Br)cc1C(=O)O)OCC1c2ccccc2-c2ccccc21,COc1nc(Cl)ncc1N,NCc1ccsc1Br,COc1nc(Cl)ncc1Nc1nc(NCc2ccsc2Br)nc(Nc2ccc(Br)c...,BRD4,0
3,256491311,O=C(O)C[C@@H](NC(=O)OCC1c2ccccc2-c2ccccc21)c1c...,Cl.Cl.NCC1CCN(CC(F)F)CC1,Cc1cn(-c2cc(N)cc(C(F)(F)F)c2)cn1,Cc1cn(-c2cc(Nc3nc(NCC4CCN(CC(F)F)CC4)nc(N[C@H]...,sEH,0
4,116128128,O=C(N[C@H](CC1CCCC1)C(=O)O)OCC1c2ccccc2-c2ccccc21,Cl.NCc1ccnc(C(N)=O)c1,NCCc1ccc(N2CCOCC2)c(F)c1,NC(=O)c1cc(CNc2nc(NCCc3ccc(N4CCOCC4)c(F)c3)nc(...,BRD4,0


**Procesamiento de Caracteristicas**

Se toman los SMILEs correspondientes a cada molecula completamente ensamblada (molecule_smiles) y se generan los ECFPs para estas. Podriamos escoger diferentes radios o bits, pero 2 y 1024 son los mas estandar.

In [29]:
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import OneHotEncoder


# Se convierten los SMILEs en moleculas RDKit
df['molecule'] = df['molecule_smiles'].apply(Chem.MolFromSmiles)

#Se generan las ECFPs (Extended-Conectivity Fingerprints- Huellas Dactilares de Conectividad Extendida)
def generate_ecfp(molecule, radius=2, bits=1024):
    if molecule is None:
        return None
    return list(AllChem.GetMorganFingerprintAsBitVect(molecule, radius, nBits=bits))

df['ecfp'] = df['molecule'].apply(generate_ecfp)


Se truncaron las últimas líneas 5000 del resultado de transmisión.
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use MorganGenerator
[19:49:20] DEPRECATION WARNING: please use Morga

**Modelo de Entrenamiento**

In [44]:
import joblib

# Se realiza la codificación One-hot a los nombres de las proteinas almacenadas en protein_name
onehot_encoder = OneHotEncoder(sparse_output=False)
protein_onehot = onehot_encoder.fit_transform(df['protein_name'].values.reshape(-1, 1))

# Se combinan las ECFPs y los nombres de la proteinas codificadas
X = [ecfp + protein for ecfp, protein in zip(df['ecfp'].tolist(), protein_onehot.tolist())]
y = df['binds'].tolist()

# Se dividen los datos en conjunto de prueba y entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Se crea y entrena el Modelo Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Se guarda el modelo
#joblib.dump(rf_model, 'modelo_entrenado.pkl')

# Se realizan las predicciones  con el conjunto de prueba
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]  # Probabilidad de la clase positiva

# Se calcula la precisión media promedio
map_score = average_precision_score(y_test, y_pred_proba)
print(f"Mean Average Precision (mAP): {map_score:.2f}")

KeyError: 'protein_name'

**Almaecenamiento del Modelo**

In [34]:
import pickle
with open("/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

**Prueba del modelo con los datos de prueba**

El modelo Random Forest entrenado es usado para predecir las probabilidad de afinidad. Estas predicciones se guardan en un archivo CSV.

In [43]:
X_test_df=pd.DataFrame(X_test)
X_test_df.to_csv("/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/test_data_input.csv", index=False)
# Cargar el archivo CSV
df = pd.read_csv('/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/test_data_input.csv')

# Imprimir el contenido
#print(df)

y_test_df=pd.DataFrame(y_test)
y_test_df.to_csv("/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/test_data_target.csv", index=False)

# Cargar el archivo CSV
df = pd.read_csv('/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/test_data_target.csv')

# Imprimir el contenido
#print(df)

#Se carga el modelo previamente creado
with open('/content/drive/MyDrive/Inteligencia Artificial y Modelos/fase-2/model.pkl', 'rb') as f:
    _m = pickle.load(f)

In [38]:
#Se realiza la prediccion
_m.predict(X_test)

array([0, 1, 1, ..., 1, 1, 1])

In [39]:
_m.score(X_test, y_test)

0.8930833333333333